[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/04_llm_finetuning/rlhf_dpo.ipynb)

# 05 . RLHF, Reward Models, PPO and DPO

## Concept — RLHF & DPO (revision notes)

**What it is**
- RLHF (Reinforcement Learning from Human Feedback) aligns a language model with human preferences in three stages: (1) supervised fine-tuning (SFT), (2) train a **reward model** on pairs "response A was preferred over response B", (3) optimise the policy against that reward with RL (usually **PPO**) while staying close to the SFT/reference model with a KL penalty.
- **DPO** (Direct Preference Optimization) skips stage 2 and 3: it turns the same objective into a simple classification-style loss on preference pairs, using the policy and a frozen reference model's log-probabilities.

**Why it matters**
- SFT teaches the model to imitate; it cannot express "this answer is better than that one". Preference data is much cheaper to collect than demonstrations and captures style, safety and helpfulness.
- DPO needs no sampling loop, no reward model, no value network - far simpler and more stable than PPO-based RLHF, which is why it is popular for open-source fine-tuning.

**When to use**
- Reward model + PPO: when you can generate fresh samples cheaply and want to keep optimising against a reusable reward signal (online).
- DPO: when you already have an offline preference dataset and want a robust, cheap method.

**Math & intuition**
- Bradley-Terry: `P(y_w > y_l | x) = sigmoid(r(x,y_w) - r(x,y_l))`. Reward model loss: `-log sigmoid(r_w - r_l)`. Only reward *differences* matter.
- PPO clipped objective: with `ratio = pi_new(a|s) / pi_old(a|s)` and advantage `A`, maximise `min(ratio*A, clip(ratio, 1-eps, 1+eps)*A)`. The clip removes the incentive to move the policy far from the one that collected the data.
- KL-regularised RL objective: `max E[r(x,y)] - beta * KL(pi || pi_ref)`. Its optimum is `pi*(y|x) ∝ pi_ref(y|x) * exp(r(x,y)/beta)`.
- Rearranging gives an implicit reward `r(x,y) = beta * log(pi(y|x)/pi_ref(y|x)) + const`, and plugging it into Bradley-Terry yields the **DPO loss**: `-log sigmoid( beta*[log pi(y_w)/pi_ref(y_w) - log pi(y_l)/pi_ref(y_l)] )`.

Everything below is pure PyTorch on a tiny decoder-only model with synthetic data (no downloads).

### 1. Imports, seeds and a synthetic preference dataset for a reward model

**What this cell does (plain language):** we create a hidden "true" reward (a linear function of 8 response features) and sample pairs of responses. For each pair a human-like annotator picks the winner with probability `sigmoid(r_a - r_b)` - exactly the Bradley-Terry model - so the labels are noisy, like real human labels.

In [1]:
import torch
torch.set_num_threads(1)  # tiny models: multi-threading overhead only slows them down (and keeps runs deterministic)
import torch, torch.nn as nn, torch.nn.functional as F
import math, copy
torch.manual_seed(0)

D = 8
w_true = torch.randn(D)                      # hidden true reward direction
def true_reward_feat(x): return x @ w_true   # r(x) = w . x

N = 2000
xa, xb = torch.randn(N, D), torch.randn(N, D)
p_a_wins = torch.sigmoid(true_reward_feat(xa) - true_reward_feat(xb))   # Bradley-Terry probability
a_wins = torch.bernoulli(p_a_wins).bool()
# chosen / rejected features for each pair
x_chosen   = torch.where(a_wins[:, None], xa, xb)
x_rejected = torch.where(a_wins[:, None], xb, xa)
print("pairs:", N, "| annotator agrees with true-reward ordering:",
      ((true_reward_feat(x_chosen) > true_reward_feat(x_rejected)).float().mean().item()))


pairs: 2000 | annotator agrees with true-reward ordering: 0.8870000243186951


### 2. Bradley-Terry reward model: train with -log sigmoid(r_w - r_l)

**What this cell does (plain language):** we train a small MLP reward model to score responses. The loss only looks at the *difference* between the chosen and rejected scores, pushing chosen above rejected. We then check pairwise accuracy on held-out pairs and how well the learned scores correlate with the hidden true reward.

In [2]:
reward_model = nn.Sequential(nn.Linear(D, 32), nn.Tanh(), nn.Linear(32, 1))
opt = torch.optim.Adam(reward_model.parameters(), lr=1e-2)

def bt_loss(r_chosen, r_rejected):
    # -log sigmoid(r_w - r_l); logsigmoid is the numerically stable form
    return -F.logsigmoid(r_chosen - r_rejected).mean()

tr, te = slice(0, 1600), slice(1600, N)
for step in range(200):
    opt.zero_grad()
    loss = bt_loss(reward_model(x_chosen[tr]).squeeze(-1), reward_model(x_rejected[tr]).squeeze(-1))
    loss.backward(); opt.step()
    if step % 50 == 0: print(f"step {step:3d} loss {loss.item():.4f}")

with torch.no_grad():
    rc, rr = reward_model(x_chosen[te]).squeeze(-1), reward_model(x_rejected[te]).squeeze(-1)
    print("held-out pairwise accuracy:", (rc > rr).float().mean().item())
    xs = torch.randn(1000, D)
    corr = torch.corrcoef(torch.stack([reward_model(xs).squeeze(-1), true_reward_feat(xs)]))[0, 1]
    print("correlation of learned vs true reward:", round(corr.item(), 3))
# Note: held-out accuracy cannot reach 100% - labels are noisy by construction.


step   0 loss 0.6726
step  50 loss 0.2418


step 100 loss 0.2305
step 150 loss 0.2187


held-out pairwise accuracy: 0.8675000071525574
correlation of learned vs true reward: 0.978


### 3. Only differences matter: reward scores are shift-invariant

**What this cell does (plain language):** Bradley-Terry only depends on `r_w - r_l`, so adding any constant to every reward leaves the loss and the preference probabilities unchanged. We verify this numerically. This is why raw reward values are not comparable across reward models, and why people normalise rewards before PPO.

In [3]:
with torch.no_grad():
    base = bt_loss(rc, rr).item()
    shifted = bt_loss(rc + 123.4, rr + 123.4).item()
print("loss:", round(base, 6), "| loss after +123.4 shift:", round(shifted, 6))
print("identical up to float error:", abs(base - shifted) < 1e-3)


loss: 0.308452 | loss after +123.4 shift: 0.308452
identical up to float error: True


### 4. A tiny decoder-only language model to act as our policy

**What this cell does (plain language):** we define a 1-layer causal transformer over a 10-token vocabulary. A "prompt" is 3 tokens and the "response" is the next 5 tokens. We add helpers to sample responses autoregressively and to compute per-token log-probabilities of a response - the quantity that PPO and DPO are both built on.

In [4]:
V, P, R = 10, 3, 5          # vocab size, prompt length, response length
T = P + R

class TinyLM(nn.Module):
    def __init__(self, d=32, heads=2):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(T, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.head, self.heads = nn.Linear(d, V), heads
    def forward(self, ids):                       # ids: (B, L) -> logits (B, L, V)
        B, L = ids.shape
        h = self.tok(ids) + self.pos(torch.arange(L))
        q, k, v = self.qkv(self.ln1(h)).chunk(3, dim=-1)
        sp = lambda t: t.view(B, L, self.heads, -1).transpose(1, 2)
        a = F.scaled_dot_product_attention(sp(q), sp(k), sp(v), is_causal=True)
        h = h + self.proj(a.transpose(1, 2).reshape(B, L, -1))
        h = h + self.mlp(self.ln2(h))
        return self.head(h)

def response_token_logps(model, prompt, resp):
    # log pi(resp_t | prompt, resp_<t) for each response token -> (B, R)
    seq = torch.cat([prompt, resp], dim=1)
    logp = F.log_softmax(model(seq[:, :-1]), dim=-1)          # predictions for positions 1..T-1
    tok_logp = logp.gather(-1, seq[:, 1:, None]).squeeze(-1)   # (B, T-1)
    return tok_logp[:, P - 1:]                                 # keep only the response part

@torch.no_grad()
def sample(model, prompt):
    seq = prompt.clone()
    for _ in range(R):
        probs = F.softmax(model(seq)[:, -1], dim=-1)
        seq = torch.cat([seq, torch.multinomial(probs, 1)], dim=1)
    return seq[:, P:]

torch.manual_seed(0)
lm = TinyLM()
pr = torch.randint(0, V, (4, P)); rs = sample(lm, pr)
print("params:", sum(p.numel() for p in lm.parameters()), "| sampled responses:\n", rs)
print("per-token logps shape:", response_token_logps(lm, pr, rs).shape)


params: 13610 | sampled responses:
 tensor([[1, 8, 3, 3, 9],
        [1, 0, 0, 5, 5],
        [3, 2, 4, 1, 1],
        [3, 8, 4, 5, 3]])
per-token logps shape: torch.Size([4, 5])


### 5. Stage 1 - SFT: create the reference policy

**What this cell does (plain language):** we "supervised fine-tune" the model on synthetic data whose response tokens come from a fixed skewed distribution (prompt tokens are random). The result is our frozen **reference model** `pi_ref`; it is neither random nor good at our target behaviour. The target behaviour used throughout: the *true reward* of a response is the number of `7` tokens in it.

In [5]:
torch.manual_seed(1)
tok_dist = torch.softmax(torch.randn(V) * 0.7, dim=0)       # skewed token distribution for SFT responses
def true_reward(resp): return (resp == 7).float().sum(-1)   # count of token 7 in the response
print("SFT token distribution:", tok_dist.numpy().round(2), "| P(token 7) =", round(tok_dist[7].item(), 3))

policy = TinyLM()
opt = torch.optim.Adam(policy.parameters(), lr=3e-3)
for step in range(300):
    prompt = torch.randint(0, V, (64, P))
    resp = torch.multinomial(tok_dist.expand(64, V), R, replacement=True)
    loss = -response_token_logps(policy, prompt, resp).sum(-1).mean()   # NLL of response given prompt
    opt.zero_grad(); loss.backward(); opt.step()
print("final SFT NLL per response:", round(loss.item(), 3), "(entropy floor ~", round(R * -(tok_dist * tok_dist.log()).sum().item(), 3), ")")

ref = copy.deepcopy(policy).eval()
for p in ref.parameters(): p.requires_grad_(False)      # frozen reference

eval_prompts = torch.randint(0, V, (512, P))            # fixed eval prompts used to compare policies
def mean_reward(model):
    torch.manual_seed(123)   # same sampling noise for every model so comparisons are fair
    return true_reward(sample(model, eval_prompts)).mean().item()
print("reference mean true reward:", round(mean_reward(ref), 3), "(expected ~", round(R * tok_dist[7].item(), 3), ")")


SFT token distribution: [0.16 0.12 0.11 0.16 0.07 0.09 0.04 0.13 0.05 0.07] | P(token 7) = 0.133


final SFT NLL per response: 11.104 (entropy floor ~ 11.058 )
reference mean true reward: 0.756 (expected ~ 0.666 )


### 6. PPO clipped objective - what the clip actually does

**What this cell does (plain language):** before using it, we tabulate the per-sample clipped surrogate `min(ratio*A, clip(ratio,1-eps,1+eps)*A)` for a positive and a negative advantage. Look at where the gradient disappears: for A>0 once the ratio exceeds `1+eps` there is no further incentive to increase the probability; for A<0 once the ratio drops below `1-eps` there is none to decrease it further. The `min` keeps the pessimistic (lower) bound, so the update can never profit from moving far from the data-collecting policy.

In [6]:
eps = 0.2
def ppo_clip_obj(ratio, adv, eps=0.2):
    return torch.minimum(ratio * adv, torch.clamp(ratio, 1 - eps, 1 + eps) * adv)

ratios = torch.tensor([0.5, 0.8, 1.0, 1.2, 1.5])
print("ratio  | obj (A=+1) grad | obj (A=-1) grad")
for r in ratios:
    out = []
    for A in (1.0, -1.0):
        rr_ = r.clone().requires_grad_(True)
        o = ppo_clip_obj(rr_, torch.tensor(A)); o.backward()
        out.append((o.item(), rr_.grad.item()))
    print(f"{r.item():.1f}    | {out[0][0]:+.2f}     {out[0][1]:+.1f}      | {out[1][0]:+.2f}     {out[1][1]:+.1f}")
# A=+1: gradient is 0 for ratio>=1.2 (clipped). A=-1: gradient is 0 for ratio<=0.8 (clipped).


ratio  | obj (A=+1) grad | obj (A=-1) grad
0.5    | +0.50     +1.0      | -0.80     +0.0
0.8    | +0.80     +0.5      | -0.80     -0.5
1.0    | +1.00     +1.0      | -1.00     -1.0
1.2    | +1.20     +0.5      | -1.20     -0.5
1.5    | +1.20     +0.0      | -1.50     -1.0


### 7. A minimal PPO loop with a KL penalty to the reference

**What this cell does (plain language):** we implement the RLHF policy-optimisation stage in ~25 lines: sample responses from the current policy, score them, turn rewards into advantages (reward minus the batch mean, normalised - a simple baseline, no value network), then take several clipped-ratio gradient steps on that same batch. A per-token KL penalty `beta * (logp - logp_ref)` is subtracted from the reward so the policy cannot wander arbitrarily far from `pi_ref`. Here the reward is the true reward, standing in for a trained reward model.

In [7]:
torch.manual_seed(2)
ppo_policy = copy.deepcopy(ref)
for p in ppo_policy.parameters(): p.requires_grad_(True)
opt = torch.optim.Adam(ppo_policy.parameters(), lr=1e-3)
beta_kl, eps = 0.3, 0.2

for it in range(40):
    prompt = torch.randint(0, V, (64, P))
    resp = sample(ppo_policy, prompt)                                   # 1. collect data with current policy
    with torch.no_grad():
        old_lp = response_token_logps(ppo_policy, prompt, resp)         # log-probs of the data-collecting policy
        ref_lp = response_token_logps(ref, prompt, resp)
        reward = true_reward(resp)
        reward = reward - beta_kl * (old_lp - ref_lp).sum(-1)           # 2. KL-shaped sequence reward
        adv = (reward - reward.mean()) / (reward.std() + 1e-8)          # 3. baseline + normalisation
    for epoch in range(4):                                              # 4. several steps on the same batch
        new_lp = response_token_logps(ppo_policy, prompt, resp)
        ratio = torch.exp(new_lp - old_lp)                              # per-token importance ratio
        loss = -ppo_clip_obj(ratio, adv[:, None], eps).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    if it % 10 == 0 or it == 39:
        print(f"iter {it:2d} | batch reward {true_reward(resp).mean().item():.2f} | last ratio range [{ratio.min().item():.2f}, {ratio.max().item():.2f}]")

ppo_policy.eval()
print("mean true reward  ref:", round(mean_reward(ref), 3), "-> PPO:", round(mean_reward(ppo_policy), 3))


iter  0 | batch reward 0.69 | last ratio range [0.85, 1.34]


iter 10 | batch reward 4.27 | last ratio range [0.89, 1.81]


iter 20 | batch reward 4.23 | last ratio range [0.90, 1.50]


iter 30 | batch reward 4.41 | last ratio range [0.96, 1.41]


iter 39 | batch reward 3.97 | last ratio range [0.69, 1.06]
mean true reward  ref: 0.756 -> PPO: 4.229


### 8. The DPO loss from scratch

**What this cell does (plain language):** we write the DPO loss as a standalone function of four sequence log-probabilities (policy and reference, for the chosen and rejected response). The *implicit reward* of a response is `beta * (log pi - log pi_ref)`; DPO is Bradley-Terry on those implicit rewards. We sanity-check two properties: when the policy equals the reference the loss is exactly `log 2` (margin 0), and the loss falls as the policy raises the chosen response relative to the reference.

In [8]:
def dpo_loss(pi_c, pi_r, ref_c, ref_r, beta=0.1):
    # all inputs: summed log-prob of the full response, shape (B,)
    chosen_reward   = beta * (pi_c - ref_c)       # implicit reward of chosen
    rejected_reward = beta * (pi_r - ref_r)       # implicit reward of rejected
    margin = chosen_reward - rejected_reward
    loss = -F.logsigmoid(margin).mean()
    acc = (margin > 0).float().mean()
    return loss, acc, margin

z = torch.zeros(5)
loss0, acc0, _ = dpo_loss(z, z, z, z)
print("policy == reference  -> loss:", round(loss0.item(), 4), "| log 2 =", round(math.log(2), 4))
for delta in (0.0, 5.0, 20.0, 50.0):    # raise chosen log-prob by delta relative to reference
    l, _, m = dpo_loss(z + delta, z, z, z, beta=0.1)
    print(f"chosen logp +{delta:4.0f} vs ref -> margin {m[0].item():.2f}, loss {l.item():.4f}")


policy == reference  -> loss: 0.6931 | log 2 = 0.6931
chosen logp +   0 vs ref -> margin 0.00, loss 0.6931
chosen logp +   5 vs ref -> margin 0.50, loss 0.4741
chosen logp +  20 vs ref -> margin 2.00, loss 0.1269
chosen logp +  50 vs ref -> margin 5.00, loss 0.0067


### 9. Build a preference dataset and train the policy with DPO

**What this cell does (plain language):** we sample two responses per prompt from the reference model and let a Bradley-Terry annotator (using the true reward) choose a winner - the same recipe as real preference data collection. Then we run DPO: precompute the frozen reference log-probs once, and train a copy of the reference with the loss above. No sampling during training, no reward model, no PPO.

In [9]:
torch.manual_seed(3)
NP = 1500
prm = torch.randint(0, V, (NP, P))
ra, rb = sample(ref, prm), sample(ref, prm)
pa_wins = torch.sigmoid(true_reward(ra) - true_reward(rb))
a_w = torch.bernoulli(pa_wins).bool()
chosen   = torch.where(a_w[:, None], ra, rb)
rejected = torch.where(a_w[:, None], rb, ra)
with torch.no_grad():
    ref_c = response_token_logps(ref, prm, chosen).sum(-1)       # frozen: compute once
    ref_r = response_token_logps(ref, prm, rejected).sum(-1)
print("fraction of pairs where chosen has strictly higher true reward:",
      (true_reward(chosen) > true_reward(rejected)).float().mean().item())

def train_dpo(beta, steps=150, lr=5e-4, seed=0, verbose=False):
    torch.manual_seed(seed)
    pol = copy.deepcopy(ref)
    for p in pol.parameters(): p.requires_grad_(True)
    o = torch.optim.Adam(pol.parameters(), lr=lr)
    for s in range(steps):
        idx = torch.randint(0, NP, (128,))
        pc = response_token_logps(pol, prm[idx], chosen[idx]).sum(-1)
        pr_ = response_token_logps(pol, prm[idx], rejected[idx]).sum(-1)
        loss, acc, margin = dpo_loss(pc, pr_, ref_c[idx], ref_r[idx], beta)
        o.zero_grad(); loss.backward(); o.step()
        if verbose and s % 50 == 0:
            print(f"step {s:3d} | loss {loss.item():.4f} | reward-acc {acc.item():.2f} | margin {margin.mean().item():.3f}")
    return pol.eval()

dpo_policy = train_dpo(beta=0.1, verbose=True)   # reward-acc 0.00 at step 0 is just margin == 0 exactly (not > 0)
print("mean true reward  ref:", round(mean_reward(ref), 3), "-> DPO:", round(mean_reward(dpo_policy), 3))


fraction of pairs where chosen has strictly higher true reward: 0.4466666579246521
step   0 | loss 0.6931 | reward-acc 0.00 | margin 0.000


step  50 | loss 0.6645 | reward-acc 0.65 | margin 0.065


step 100 | loss 0.5590 | reward-acc 0.73 | margin 0.481


mean true reward  ref: 0.756 -> DPO: 4.861


### 10. Effect of beta: reward gained vs distance from the reference

**What this cell does (plain language):** `beta` controls how strongly the policy is tied to the reference: small beta lets the policy move far (higher reward, larger KL), large beta keeps it close. We train DPO with three betas and measure (a) mean true reward and (b) a Monte-Carlo estimate of `KL(pi || pi_ref)` from sampled responses. This trade-off is the heart of KL-regularised alignment.

In [10]:
@torch.no_grad()
def kl_to_ref(model, n=512):
    torch.manual_seed(321)
    pm = torch.randint(0, V, (n, P)); rs_ = sample(model, pm)
    return (response_token_logps(model, pm, rs_) - response_token_logps(ref, pm, rs_)).sum(-1).mean().item()

print(f"{'model':<12}{'mean reward':>12}{'KL(pi||ref)':>14}")
print(f"{'reference':<12}{mean_reward(ref):>12.3f}{0.0:>14.3f}")
for b in (0.02, 0.1, 0.5):
    m = train_dpo(beta=b)
    print(f"{'DPO b='+str(b):<12}{mean_reward(m):>12.3f}{kl_to_ref(m):>14.3f}")
print(f"{'PPO':<12}{mean_reward(ppo_policy):>12.3f}{kl_to_ref(ppo_policy):>14.3f}")


model        mean reward   KL(pi||ref)
reference          0.756         0.000


DPO b=0.02         4.992        10.099


DPO b=0.1          4.861         9.328


DPO b=0.5          2.479         1.981
PPO                4.229         6.922


### 11. The implicit reward: DPO policy as a hidden reward model

**What this cell does (plain language):** DPO's derivation says `beta * log(pi/pi_ref)` is a reward model in disguise. We score fresh responses with it and check the correlation with the true reward (count of `7`s). If DPO worked, responses with more 7s should have a higher implicit reward.

In [11]:
with torch.no_grad():
    pm = torch.randint(0, V, (1000, P)); rs_ = sample(ref, pm)
    implicit = 0.1 * (response_token_logps(dpo_policy, pm, rs_).sum(-1) - response_token_logps(ref, pm, rs_).sum(-1))
    tr_ = true_reward(rs_)
    print("corr(implicit reward, true reward):", round(torch.corrcoef(torch.stack([implicit, tr_]))[0, 1].item(), 3))
    for k in range(0, int(tr_.max().item()) + 1):
        sel = tr_ == k
        if sel.sum() > 5: print(f"  #7s = {k}: n={int(sel.sum()):4d}, mean implicit reward {implicit[sel].mean().item():+.3f}")


corr(implicit reward, true reward): 0.965
  #7s = 0: n= 531, mean implicit reward -2.796
  #7s = 1: n= 343, mean implicit reward -1.847
  #7s = 2: n= 107, mean implicit reward -0.997
  #7s = 3: n=  17, mean implicit reward -0.197


## Common bugs

- **Forgetting to freeze / detach the reference model**
  If `pi_ref` gets gradients (or is the same object as the policy) the 'anchor' moves with the policy and DPO/KL terms silently become zero. Fix: `deepcopy` the SFT model, `requires_grad_(False)`, and compute reference log-probs under `torch.no_grad()` (ideally once).

- **Using `log(sigmoid(x))` instead of `logsigmoid`**
  For large negative margins `sigmoid` underflows to 0 and `log(0) = -inf`/NaN. Fix: always use `F.logsigmoid` (as in the BT and DPO losses above).

- **Summing vs averaging token log-probs inconsistently**
  DPO is defined on the log-probability of the *whole sequence* (sum over tokens). Averaging per token changes the effective beta and length bias; mixing the two between policy and reference is a plain bug. Fix: sum over response tokens only and be consistent.

- **Including prompt tokens in the log-prob**
  The prompt is the same for chosen and rejected and cancels in theory, but with padding and truncation it adds noise (and in PPO would reward the model for 'predicting' the prompt). Fix: mask everything before the response, as `response_token_logps` does.

- **Not detaching the old log-probs in PPO**
  If `old_lp` carries gradients the ratio is always 1 in value but its gradient is wrong, so the clip never engages. Fix: compute `old_lp` under `torch.no_grad()` once per batch, before the inner epochs.

- **Reward hacking / no KL penalty**
  Optimising a learned reward without a KL leash makes the policy find degenerate outputs the reward model over-scores. Fix: keep `beta > 0`, monitor KL to the reference, and evaluate with held-out or human judgements.

- **Preference labels in the wrong order**
  Swapping chosen and rejected makes BT/DPO *minimise* the quality you wanted, and training loss still goes down. Fix: unit-test with a tiny set where the answer is known; track pairwise accuracy against a held-out ground truth.

- **Dropout left on when computing reference/policy log-probs**
  Dropout makes two forward passes of the same model disagree, so the ratio at the first PPO step is not 1. Fix: use `model.eval()` for log-prob computation (or disable dropout) during RL/DPO.

## Exercises

Attempt each one in the empty cell right after the question. My full solution is in the cell after that - don't peek until you've tried.

**Exercise 1.** Verify empirically that BT preference labels are calibrated: bin the true reward gap `r_a - r_b` of the pairs in section 1 and show that the empirical win rate of `a` in each bin matches `sigmoid(gap)`.

In [12]:
# Your attempt for Exercise 1 here


**Solution 1**

In [13]:
# Solution 1
gap = (true_reward_feat(xa) - true_reward_feat(xb))
edges = torch.tensor([-4., -2., -1., 0., 1., 2., 4.])
bins = torch.bucketize(gap, edges)
for b in range(1, len(edges)):
    sel = bins == b
    if sel.sum() > 0:
        print(f"gap bin ({edges[b-1].item():+.0f},{edges[b].item():+.0f}] n={int(sel.sum()):4d} empirical {a_wins[sel].float().mean().item():.2f} vs sigmoid(mean gap) {torch.sigmoid(gap[sel].mean()).item():.2f}")


gap bin (-4,-2] n= 247 empirical 0.06 vs sigmoid(mean gap) 0.05
gap bin (-2,-1] n= 154 empirical 0.15 vs sigmoid(mean gap) 0.18
gap bin (-1,+0] n= 170 empirical 0.44 vs sigmoid(mean gap) 0.38
gap bin (+0,+1] n= 160 empirical 0.62 vs sigmoid(mean gap) 0.62
gap bin (+1,+2] n= 178 empirical 0.83 vs sigmoid(mean gap) 0.82
gap bin (+2,+4] n= 265 empirical 0.93 vs sigmoid(mean gap) 0.95


**Exercise 2.** Add L2 reward regularisation `0.01 * (r_w**2 + r_l**2).mean()` to the reward model loss and compare the scale of the scores with and without it.

In [14]:
# Your attempt for Exercise 2 here


**Solution 2**

In [15]:
# Solution 2
def train_rm(reg):
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(D, 32), nn.Tanh(), nn.Linear(32, 1))
    o = torch.optim.Adam(m.parameters(), lr=1e-2)
    for _ in range(200):
        rc_, rr_ = m(x_chosen[tr]).squeeze(-1), m(x_rejected[tr]).squeeze(-1)
        loss = bt_loss(rc_, rr_) + reg * (rc_**2 + rr_**2).mean()
        o.zero_grad(); loss.backward(); o.step()
    with torch.no_grad(): return m(torch.randn(1000, D)).std().item()
print("score std without reg:", round(train_rm(0.0), 3), "| with reg:", round(train_rm(0.01), 3))


score std without reg: 3.869 | with reg: 1.796


**Exercise 3.** Change the PPO clip range `eps` to 0.02 and 0.5 in the table function. At what ratio does the gradient vanish for A>0 in each case?

In [16]:
# Your attempt for Exercise 3 here


**Solution 3**

In [17]:
# Solution 3
for e in (0.02, 0.2, 0.5):
    r = torch.tensor(1.0 + e + 0.01, requires_grad=True)
    ppo_clip_obj(r, torch.tensor(1.0), e).backward()
    print(f"eps={e}: gradient at ratio {r.item():.2f} (just above 1+eps) = {r.grad.item()}")


eps=0.02: gradient at ratio 1.03 (just above 1+eps) = 0.0
eps=0.2: gradient at ratio 1.21 (just above 1+eps) = 0.0
eps=0.5: gradient at ratio 1.51 (just above 1+eps) = 0.0


**Exercise 4.** Compute the DPO loss gradient w.r.t. the chosen log-prob at policy == reference. It should equal `-beta * 0.5`. Verify with autograd.

In [18]:
# Your attempt for Exercise 4 here


**Solution 4**

In [19]:
# Solution 4
pc = torch.zeros(1, requires_grad=True)
l, _, _ = dpo_loss(pc, torch.zeros(1), torch.zeros(1), torch.zeros(1), beta=0.1)
l.backward()
print("autograd:", pc.grad.item(), "| expected -beta * sigmoid(0) =", -0.1 * 0.5)


autograd: -0.05000000074505806 | expected -beta * sigmoid(0) = -0.05


**Exercise 5.** Implement the *length-normalised* DPO variant (divide each sequence log-prob by response length) and train with beta=0.1. Does the reward still improve?

In [20]:
# Your attempt for Exercise 5 here


**Solution 5**

In [21]:
# Solution 5
def train_dpo_ln(beta=0.1, steps=100):
    torch.manual_seed(0)
    pol = copy.deepcopy(ref)
    for p in pol.parameters(): p.requires_grad_(True)
    o = torch.optim.Adam(pol.parameters(), lr=5e-4)
    for _ in range(steps):
        idx = torch.randint(0, NP, (128,))
        pc = response_token_logps(pol, prm[idx], chosen[idx]).sum(-1) / R
        pr_ = response_token_logps(pol, prm[idx], rejected[idx]).sum(-1) / R
        loss, _, _ = dpo_loss(pc, pr_, ref_c[idx] / R, ref_r[idx] / R, beta)
        o.zero_grad(); loss.backward(); o.step()
    return pol.eval()
print("reward ref:", round(mean_reward(ref), 3), "-> length-normalised DPO:", round(mean_reward(train_dpo_ln()), 3))


reward ref: 0.756 -> length-normalised DPO: 4.947


**Exercise 6.** Flip the labels (swap chosen/rejected) and run DPO. Show that the mean true reward goes DOWN, illustrating the label-order bug.

In [22]:
# Your attempt for Exercise 6 here


**Solution 6**

In [23]:
# Solution 6
c_bak, r_bak, rc_bak, rr_bak = chosen, rejected, ref_c, ref_r
chosen, rejected, ref_c, ref_r = r_bak, c_bak, rr_bak, rc_bak     # swap everything
flipped = train_dpo(beta=0.1)
chosen, rejected, ref_c, ref_r = c_bak, r_bak, rc_bak, rr_bak     # restore
print("reward ref:", round(mean_reward(ref), 3), "-> flipped-label DPO:", round(mean_reward(flipped), 3))


reward ref: 0.756 -> flipped-label DPO: 0.0
